# molcore quickstart

**AI-native cheminformatics** — Rust performance core · RDKit bridge · PyG/PyTorch integration

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anteneh-T-Tessema/molcore/blob/main/examples/quickstart.ipynb)

This notebook walks through the full molcore workflow in ~10 minutes:
1. Parse molecules and render 2D structures
2. Batch fingerprints (Rust-accelerated) and Tanimoto screening
3. Molecular descriptors and Lipinski filtering
4. Load/save SDF files and Parquet datasets
5. Scaffold-based train/val/test split
6. Train a GCN property predictor with uncertainty estimates
7. SMARTS substructure search, MCS, and R-group decomposition
8. Standardization pipeline

In [ ]:
# Colab install (skip if running locally)
import sys
if 'google.colab' in sys.modules:
    !pip install molcore torch --index-url https://download.pytorch.org/whl/cpu -q
    !pip install torch-geometric pyarrow pandas -q

## 1. Molecules and 2D rendering

In [ ]:
from molcore.molecule import Mol

# Immutable — transforms always return new Mol instances
aspirin = Mol.from_smiles("CC(=O)Oc1ccccc1C(=O)O")
print(aspirin)

# Jupyter renders 2D structure automatically via _repr_svg_
aspirin

In [ ]:
# Transforms return NEW Mol — aspirin is unchanged
neutral = aspirin.neutralize()
clean   = aspirin.standardize()   # strip salts → neutralize → canonical tautomer

print("original :", aspirin.smiles)
print("clean    :", clean.smiles)

In [ ]:
# Grid of molecules
from molcore.rdkit_bridge import mols_to_grid_svg
from IPython.display import SVG

smiles = [
    "CC(=O)Oc1ccccc1C(=O)O",   # aspirin
    "CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C",  # testosterone
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",          # caffeine
    "CC(=O)Nc1ccc(O)cc1",                    # paracetamol
    "OC(=O)c1ccccc1O",                        # salicylic acid
    "CC(C)Cc1ccc(cc1)C(C)C(=O)O",            # ibuprofen
    "c1ccc2c(c1)cc1ccc3cccc4ccc2c1c34",      # pyrene
    "O=C(O)c1ccc(N)cc1",                     # 4-aminobenzoic acid
]

names = ["aspirin", "testosterone", "caffeine", "paracetamol",
         "salicylic acid", "ibuprofen", "pyrene", "PABA"]

SVG(mols_to_grid_svg(smiles, mols_per_row=4, legends=names))

## 2. Batch fingerprints and Tanimoto screening

In [ ]:
from molcore.pipeline import featurize_smiles

# Rust Rayon parallel — zero-copy numpy → torch
fps = featurize_smiles(smiles, backend="rust")  # (8, 2048) uint8 Tensor
print(f"Shape: {fps.shape} | dtype: {fps.dtype}")

# Other fingerprint types
maccs = featurize_smiles(smiles, kind="maccs")          # (8, 167)
ap    = featurize_smiles(smiles, kind="atom_pairs")     # (8, 2048)
print(f"MACCS: {maccs.shape} | AtomPairs: {ap.shape}")

In [ ]:
from molcore._molcore import tanimoto_matrix
import torch

# Query: aspirin only.  Library: all 8 molecules
q_fps = fps[:1].numpy()   # (1, 2048)
l_fps = fps.numpy()       # (8, 2048)

sim = tanimoto_matrix(q_fps, l_fps)   # (1, 8) float32

print("Tanimoto similarity to aspirin:")
for name, score in zip(names, sim[0]):
    print(f"  {name:<20} {score:.3f}")

## 3. Descriptors and Lipinski filtering

In [ ]:
from molcore.rdkit_bridge import calc_named_descriptors
import pandas as pd
import numpy as np

arr, cols = calc_named_descriptors(smiles, preset="lipinski")

desc_df = pd.DataFrame(arr, columns=cols, index=names)
desc_df = desc_df.round(2)
desc_df

In [ ]:
# Lipinski Ro5 filter
ro5 = (
    (desc_df["MolWt"]           <= 500) &
    (desc_df["MolLogP"]         <=   5) &
    (desc_df["NumHDonors"]      <=   5) &
    (desc_df["NumHAcceptors"]   <=  10)
)
print("Ro5 compliant:", desc_df[ro5].index.tolist())
print("Ro5 violators :", desc_df[~ro5].index.tolist())

## 4. SDF and Parquet I/O

In [ ]:
import tempfile, os
from molcore.io import MolDataset

# Build a dataset with fingerprints and descriptors
ds = MolDataset.from_smiles(smiles, compute_fps=True, compute_desc=True)
ds.labels = arr[:, 0]   # use MW as a dummy label
ds.metadata["name"] = names
print(ds)

# Write to SDF
with tempfile.TemporaryDirectory() as d:
    sdf_path = os.path.join(d, "library.sdf")
    ds.write_sdf(sdf_path)

    # Load back
    ds2 = MolDataset.from_sdf(sdf_path, compute_fps=True, compute_desc=True)
    print(f"Loaded {len(ds2)} molecules from SDF")

# Jupyter inline grid (auto-renders via _repr_html_)
ds

In [ ]:
# Pandas bridge
import molcore.pandas_tools as mpt

df = pd.DataFrame({"smiles": smiles, "name": names})
df = mpt.add_mol_column(df)            # add 'Mol' column
df = mpt.add_descriptors(df, preset="lipinski")  # add descriptor columns
df = mpt.add_fingerprints(df, kind="ecfp4")      # add 'fp' column
df[["name", "MolWt", "MolLogP", "TPSA"]].round(2)

## 5. Scaffold split

In [ ]:
# Larger example: use 50 SMILES from a built-in list
from molcore.io import MolDataset

# Reuse our 8 — small but illustrates the API
ds = MolDataset.from_smiles(smiles, compute_fps=False, compute_desc=False)
ds.labels = arr[:, 0]

train, val, test = ds.scaffold_split(train_frac=0.6, val_frac=0.2, seed=42)
print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")

## 6. GCN property predictor with MC Dropout uncertainty

In [ ]:
from molcore.predictor import PropertyPredictor
from molcore.io import MolDataset
import numpy as np

# Predict log P for a small set (real use: 1k+ molecules)
train_smiles = [
    "CCO", "CC(=O)O", "c1ccccc1", "c1ccccc1O", "CC(C)O",
    "CCCO", "CCCCO", "CCc1ccccc1", "Cc1ccccc1", "c1ccc(O)cc1",
]
logp_values = np.array([-0.14, -0.17, 1.68, 1.46, 0.05,
                          0.25,  0.88,  2.27,  2.11,  1.46], dtype=np.float32)

ds = MolDataset.from_smiles(train_smiles, compute_fps=False, compute_desc=False)
ds.labels = logp_values

pred = PropertyPredictor(hidden=32, n_layers=2, epochs=80, lr=5e-3)
pred.fit(ds, verbose=False)

test_smiles = ["CCC", "CCCc1ccccc1", "c1ccc(N)cc1"]
predictions = pred.predict(test_smiles)
print("Predictions:", predictions.round(2))

In [ ]:
# MC Dropout uncertainty — no retraining needed
means, stds = pred.predict_with_uncertainty(test_smiles, n_samples=50)

result_df = pd.DataFrame({
    "smiles": test_smiles,
    "logp_predicted": means.round(3),
    "uncertainty_std": stds.round(3),
})
result_df

## 7. Substructure search, MCS, and R-group decomposition

In [ ]:
from molcore.rdkit_bridge import filter_by_smarts, find_mcs, rgroup_decompose

# Substructure filter — keep aromatic compounds
aromatic = filter_by_smarts(smiles, "a1aaaaa1")
print(f"Aromatic: {len(aromatic)} / {len(smiles)}")

# Filter directly on Mol object
mol = Mol.from_smiles("c1cccnc1CC")
print("Contains pyridine:", mol.matches("c1ccncc1"))

In [ ]:
# MCS — find the largest shared substructure in a series of analogs
analogs = [
    "CC(=O)Oc1ccccc1",        # phenyl acetate
    "CC(=O)Oc1ccc(F)cc1",     # 4-F analog
    "CC(=O)Oc1ccc(Cl)cc1",    # 4-Cl analog
    "CC(=O)Oc1ccc(Br)cc1",    # 4-Br analog
]
mcs_smarts = find_mcs(analogs)
print("MCS SMARTS:", mcs_smarts)

In [ ]:
# R-group decomposition — extract R-groups from a para-substituted benzene series
core = "c1ccc([*:1])cc1"   # para-substituted benzene
series = ["c1ccc(F)cc1", "c1ccc(Cl)cc1", "c1ccc(Br)cc1", "c1ccc(N)cc1", "CCO"]

rows = rgroup_decompose(core, series)
rg_df = pd.DataFrame([
    {"smiles": s, **r} for s, r in zip(series, rows)
])
rg_df

## 8. Standardization pipeline

In [ ]:
from molcore.rdkit_bridge import standardize

messy = [
    "[Na+].OC(=O)c1ccccc1",     # sodium salt
    "CC(=O)[O-]",                # charged
    "OCC",                       # non-canonical SMILES
    "[H]C1=CC=CC=C1",            # explicit H
]

clean = [standardize(s) for s in messy]

for orig, std in zip(messy, clean):
    print(f"  {orig:<30}  →  {std}")

In [ ]:
# Or via pandas_tools on a full DataFrame
import molcore.pandas_tools as mpt

df_messy = pd.DataFrame({"smiles": messy})
df_clean = mpt.standardize_smiles(df_messy)
df_clean

## 9. PyG Data objects — ready for GNN training

Every `Mol` exports zero-copy to PyG `Data` with 9 node features.

In [ ]:
mol = Mol.from_smiles("CC(=O)Oc1ccccc1C(=O)O")  # aspirin
data = mol.to_pyg()

print("Node features  x:         ", data.x.shape, data.x.dtype)
print("Edge index     edge_index:", data.edge_index.shape, data.edge_index.dtype)
print("Edge features  edge_attr: ", data.edge_attr.shape, data.edge_attr.dtype)
print()
print("Node feature layout:")
print("  [0] atomic_num | [1] is_aromatic | [2] formal_charge | [3] num_hs")
print("  [4] degree     | [5] in_ring     | [6] hybridization | [7] chirality | [8] mass_norm")

In [ ]:
# Heterogeneous graph — partitioned by atom type
hetero = mol.to_pyg_hetero()
print("HeteroData node types:", hetero.node_types)

---

## Next steps

- **[Benchmarks](../benchmarks/bench_fingerprints.py)** — ECFP4 throughput vs RDKit
- **[End-to-end GNN example](end_to_end_gnn.py)** — train a GCN on ESOL solubility
- **[Migration guide](../docs/migrating_from_rdkit.md)** — port existing RDKit scripts
- **[Skills](../skills/)** — agent-ready skill definitions for each capability

For questions or issues: https://github.com/Anteneh-T-Tessema/molcore/issues